# STAT 626 — 100 patient groups, true 2D cross-attention

Upload this notebook at https://colab.research.google.com/ using **File → Upload notebook**.
Select **Runtime → Change runtime type → GPU**, then run the cells in order.
The notebook embeds the complete source and the frozen selection of **100 groups**.

The saved local cohort contains 1,251 examinations and 1,133 filename-derived groups.
We select one complete examination from each of 100 groups using seed 626, before
reading segmentation labels. After QC the target split is **70 train / 15 validation / 15 test**.
The identity grouping comes from filenames and has not been independently verified.

**True 2D:** one aligned axial slice from each of T1, T1ce, T2 and FLAIR, plus
four availability maps: **8 × H × W** (240 × 240 in the local examples).
Predict that same axial slice; no neighboring-slice inputs or 3D convolutions.
The NIfTI files remain 3D storage containers. At evaluation, independent 2D predictions
are stacked to compute patient-volume Dice and HD95.

**Small study:** A0 masked mean fusion versus A1 cross-modal attention, width 16,
10 epochs, one seed (11), 16 sampled slices per training patient per epoch.
No lambda grid or regularization term is needed for this comparison. A2 remains optional.
The earlier code was 2.5D with 2D convolutions; this version removes its neighboring-slice context.

Start with QUICK_CHECK to test execution. Then switch to STUDY_100_2D for the 100-group study.
One seed and 15 test groups provide an exploratory comparison, not a definitive performance claim.


## 1. Install dependencies and verify the GPU
Use Colab's existing CUDA-enabled PyTorch. This cell does not replace it with a CPU build.


In [ ]:
import os, sys, subprocess, importlib.metadata
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["MPLBACKEND"] = "Agg"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "nibabel>=5.2,<6", "numpy>=1.26,<3", "scipy>=1.11,<2",
                "matplotlib>=3.8,<4"], check=True)
import torch, numpy as np
from packaging.version import Version
if Version(torch.__version__.split("+")[0]) < Version("2.3"):
    raise RuntimeError("Use a current Colab GPU runtime with PyTorch >= 2.3.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Choose Runtime > Change runtime type > GPU, then rerun.")
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
ENVIRONMENT = {"python": sys.version.split()[0], "torch": str(torch.__version__),
               "cuda": torch.version.cuda,
               **{name: importlib.metadata.version(name) for name in ("numpy", "nibabel", "scipy", "matplotlib")}}


## 2. Mount Drive and choose the run
The default source is your earlier folder containing 52 ZIP shards and
`BraTS_Full_Manifest.json`. The notebook extracts **only the selected 100 examinations**.
Reading/copying the relevant original shards can still transfer much of the source archive data.

For a smaller upload, upload the supplied **BraTS100_2D.zip** to `My Drive/STAT626`.
The notebook will prefer it automatically when present. An extracted cohort folder or
an explicit ZIP path also works. Hosted Colab cannot read your Windows D: drive.
If your earlier BraTS200_2D.zip is already on Drive, set DATA_SOURCE to its full path;
this notebook will extract only the selected 100 examinations from it.

| Profile | Audited scans | Models | Width | Epochs | Seeds |
|---|---:|---|---:|---:|---|
| QUICK_CHECK | First six selected scans | A0, A1 | 4 | 1 | 11 |
| STUDY_100_2D | All 100 selected scans | A0, A1 | 16 | 10 | 11 |

For a longer comparison or more seeds, edit the settings in a new study folder before
training. Use the same settings for both models and avoid changing them after viewing test scores.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import json, copy

PROFILE = "QUICK_CHECK" # @param ["QUICK_CHECK", "STUDY_100_2D"]
DATA_SOURCE = "/content/drive/MyDrive/STAT626" # @param {type:"string"}
STUDY_BASE = "/content/drive/MyDrive/STAT626_100patients_2D_v1" # @param {type:"string"}
PATIENT_MAP = "" # @param {type:"string"}
LOCAL_DATA_ROOT = "/content/stat626_100_2d_data" # @param {type:"string"}
STUDY_DIR = Path(STUDY_BASE) / PROFILE.lower()
CODE_DIR = Path("/content/stat626_100_2d_code")
if not Path(DATA_SOURCE).exists():
    raise FileNotFoundError(f"Correct DATA_SOURCE or upload the subset ZIP first: {DATA_SOURCE}")
print("Profile:", PROFILE, "Persistent outputs:", STUDY_DIR)


## 3. Load the embedded project code
No GitHub repository or separate code upload is needed. The source is also supplied
as readable `.py` files beside this notebook. This cell restores the packaged version.
If you edit the separate Python files, you must load those edited files explicitly
instead of running this embedded-code cell.


In [ ]:
# @title Restore true-2D source and the frozen 100-group selection
import base64, hashlib, io, zipfile, importlib
CODE_B64 = "..."
payload = base64.b64decode(CODE_B64)
assert hashlib.sha256(payload).hexdigest() == "01c557f50462f08ad8335355408569ec782bbec0643ab6e9180e945138cea84c"
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert set(archive.namelist()) == {'design_network.py', 'train_model.py', 'colab_workflow.py', 'subset_selection.json', 'eda.py', 'select_subset.py', 'requirements_stat626.txt'}
    archive.extractall(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for module_name in ("eda", "design_network", "train_model", "colab_workflow", "select_subset"):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
import colab_workflow as workflow
from select_subset import stage_subset
SETTINGS = copy.deepcopy(workflow.PROFILES[PROFILE])
SETTINGS.update(data_source=DATA_SOURCE, local_data_root=LOCAL_DATA_ROOT,
                expected_groups=100, patient_map=PATIENT_MAP,
                patient_map_sha256=workflow.digest(PATIENT_MAP) if PATIENT_MAP else None)
workflow.lock_study(STUDY_DIR, CODE_DIR, PROFILE, SETTINGS, ENVIRONMENT)
print(json.dumps(SETTINGS, indent=2))
selection = json.loads((CODE_DIR / "subset_selection.json").read_text())
print("Frozen selected groups:", selection["selected_patient_groups"])


## 4. Stage only the selected examinations
This keeps 100 examinations on local runtime storage. The original dataset remains unchanged.
For the original shard format, SHA-256 hashes are verified before extracting selected files.
For a single ZIP, ZIP CRC is checked during extraction. The selection JSON fixes case IDs and sizes.
After a reset, rerun staging and training cells; your trained epoch checkpoints remain on Drive.


In [ ]:
DATA_ROOT = stage_subset(DATA_SOURCE, LOCAL_DATA_ROOT, CODE_DIR / "subset_selection.json")
print("Selected data staged:", DATA_ROOT)


## 5. Run EDA, quality control, and freeze patient splits
Saved audits are reused with integrity checks. A partial QUICK_CHECK audit is never
used as the full study audit. Inspect data counts, naturally absent files, dimensions,
and the distinction between acquired scans and simulated missing-input views.


In [ ]:
EDA_DIR, EDA_SUMMARY = workflow.audit(CODE_DIR, DATA_ROOT, STUDY_DIR, SETTINGS, PATIENT_MAP)
for key in ("audit_scope", "data_volume", "retained_scan_folders", "retained_patient_groups",
            "retained_slice_dimensions", "split_scan_counts", "split_patient_counts"):
    print(key, ":", json.dumps(EDA_SUMMARY.get(key), indent=2))
from IPython.display import display, Image
for name in ("segmentation_explained.png", "single_slice_explained.png"):
    path = EDA_DIR / name
    if path.exists():
        display(Image(filename=str(path), width=1000))


# Literature-based design for the 100-group, 2D project

## Research question

**Does learning interactions between available MRI modalities improve segmentation over matched mean fusion, and does complete/missing consistency provide an additional benefit?**

Treat each MRI sequence as an information source: its image supplies features, its identity tells the model which sequence produced those features, and its availability indicator tells the model whether it is present. An availability flag alone does not contain the missing image's information. Cross-attention combines observed features; it does not recover a genuinely acquired missing scan.

## Review of the five suggested papers

| Paper and source | Relevant contribution | Use in our simplified project | Scope of the adaptation |
|---|---|---|---|
| [CKD-TransBTS, Lin et al., 2022 preprint](https://arxiv.org/html/2207.07370v1) | Groups T1/T1ce and T2/FLAIR; combines a CNN/transformer encoder with modality-correlated cross-attention and decoder feature calibration. Evaluated on BraTS 2021. | Encode sequence information before fusion; learn interactions between streams. | We use four separate 2D stems and local modality attention, without its 3D hybrid encoder or feature-calibration decoder. |
| [CMAF-Net, Sun et al., 2024](https://qims.amegroups.org/article/view/125898/html) | Targets incomplete multimodal segmentation using modality-specific encoding, 3D Swin processing, cross-modal fusion and decoder attention. Training simulates missing inputs through modality dropout. | Closest task match: combine available streams and test missing-modality conditions explicitly. | Our masked 2D module is smaller. We test complete input and four single-missing conditions, not every nonempty modality subset. |
| [Clinical Knowledge-Based Hybrid Swin Transformer, Lei et al., 2023](https://www.techscience.com/cmc/v76n3/54381/html) | Pairs T1/T1ce and T2/FLAIR, using CNN/Swin branches and cross-attention between modality groups on BraTS 2021. | Supports structured fusion of complementary sequences. | Our model learns interactions among four individual streams; paired-group fusion remains a possible separate ablation. |
| [MSegNet, Wang et al., 2025](https://link.springer.com/article/10.1007/s44196-025-00787-7) | Combines 3D convolutional encoding/decoding with cross-modal transformer processing and multi-view spatial/depth context. | Motivates learning inter-modal relationships. | We retain only single-slice, local cross-modal interaction; we sacrifice its 3D/multi-view context. |
| [CMHF-3DNet, Hussain et al., 2026](https://pubmed.ncbi.nlm.nih.gov/42225491/) | The authors' abstract describes voxel-wise cross-modal transformer fusion and hierarchy-aware multitask learning for ET contained in TC contained in WT. | Explain tumor-region nesting and maintain consistent evaluation labels. | Our four exclusive tissue labels already produce nested WT/TC/ET unions. No extra hierarchy penalty is added. |

The first four rows use the primary preprint/publisher methods pages. For CMHF-3DNet, the publisher full text could not be retrieved (403); its review is limited to the authors' abstract indexed by PubMed. Its exact attention implementation and reproducibility have not been assessed. MSegNet has a [published correction to its funding statement](https://link.springer.com/article/10.1007/s44196-025-00807-6).

Published results establish motivation for testing these ideas. Differences in cohorts, splits, missingness conditions and 3D versus 2D architecture prevent a direct numerical ranking against our experiment. We are not claiming to reproduce these five systems or outperform them.

## Our architecture, aligned with the whiteboard sketch

For one axial location z, use four registered images of size 240 × 240, one per modality. Four constant availability maps make the input 8 × 240 × 240. The network predicts four tissue-class logits at each pixel of that same slice.

```mermaid
flowchart LR
  T1[T1 slice] --> E1[2D stem 1]
  T1C[T1ce slice] --> E2[2D stem 2]
  T2[T2 slice] --> E3[2D stem 3]
  F[FLAIR slice] --> E4[2D stem 4]
  E1 --> M[Mask unavailable streams]
  E2 --> M
  E3 --> M
  E4 --> M
  M --> A[Mean fusion or cross-attention]
  A --> U[2D U-Net]
  U --> O[Four tissue classes]
```

At a corresponding coarse spatial location, let h_m be a modality feature token after normalization and its learned modality embedding. A query from modality m attends to other available modalities:

\[
Q_m=h_mW_Q,\quad K_n=h_nW_K,\quad V_n=h_nW_V,
\]
\[
C_m=\operatorname{softmax}\left(\frac{Q_mK_{-m}^{T}}{\sqrt{d_k}}+M\right)V_{-m}.
\]

M excludes unavailable keys and the query's own modality. With only one available modality, the implementation uses a finite self fallback; no available modalities is an error. In the planned experiments there are three or four available streams. The multi-head outputs pass through residual/feed-forward processing, are averaged across available streams, and update the fused feature map before the U-Net.

In this implementation, attention connects modalities at each coarse image location. It does not connect every pixel to every other pixel. The 2D U-Net handles spatial processing, while the attention module handles relationships among the sequences. No neighboring axial slices enter the network.

## Cross-attention and regularization answer different questions

Cross-attention changes how features are combined. Consistency regularization changes what the training objective encourages. They can be tested independently within a controlled comparison.

The common supervised objective is:

\[
L_{sup}=\tfrac12\left[L_{seg}(p_{full},y)+L_{seg}(p_{missing},y)\right],
\qquad L_{seg}=L_{CE}+L_{Dice}.
\]

An optional consistency term encourages the incomplete-input prediction to agree with the complete-input prediction over the fixed brain mask B:

\[
L_{cons}=\frac{1}{|B|}\sum_{v\in B}
D_{KL}\left(\operatorname{stopgrad}(p_{full,v})\;\|\;p_{missing,v}\right),
\qquad L=L_{sup}+\lambda L_{cons}.
\]

This is the project's consistency design; it is not attributed to all five papers. The complete-view distribution is detached for this term, but its supervised loss still trains the shared model. A large lambda could enforce an incorrect or overconfident prediction, so its benefit must be measured rather than assumed.

| Model | Fusion | Objective | Comparison |
|---|---|---|---|
| A0 | Available-stream mean | L_sup | Matched baseline |
| A1 | Cross-modal attention | L_sup | A1−A0 tests attention |
| A2, optional | Same attention as A1 | L_sup + lambda L_cons | A2−A1 tests consistency |

The optional notebook experiment fixes lambda=0.1 before evaluation to keep the study small. This is a preset, not an optimized value. A later lambda search must use validation patients only and be reported as additional tuning. A0, A1 and A2 in that experiment use the same 100-group cohort, 70/15/15 split, width 16, seed 11, 10-epoch cap and sampling/augmentation schedule.

The optional experiment uses its own results folder and reruns all three models under the same settings. It does not reuse an incompatible earlier 2.5D checkpoint. A0 and A1 have 1,946,104 and 1,946,736 parameters respectively; adding attention also changes computation and parameter count, even with shared stems/backbone.

## What would count as evidence of improvement?

Freeze the selection, patient split and protocol before opening test results. The main exploratory outcome is the paired patient-level A1−A0 Dice difference averaged across WT/TC/ET and the four single-missing conditions. Also report each region/condition, complete-input performance, HD95, runtime and memory. The optional A2−A1 contrast asks whether consistency adds value within attention.

Use the same held-out groups and evaluate all axial slices; stack predictions for patient-volume metrics. Do not count slices or the five views of one examination as independent patients. A 15-patient bootstrap is conditional on this split and trained seed; it does not capture training-seed variability. Report the experiment as exploratory, including intervals that cross zero and any condition where attention performs worse. Preselect any extension before inspecting test results; an extension chosen after seeing those scores remains exploratory and cannot claim a fresh independent confirmation.

## Presentation wording

“The literature motivates learning interactions between MRI sequences. We adapt this idea to a small 2D study with 100 patient groups. A matched mean-fusion model provides the baseline, cross-attention tests whether learned modality interactions help, and an optional consistency penalty tests robustness to missing inputs. We will assess improvement on the same held-out patients rather than assume that attention is beneficial.”

## Citation details

- CKD-TransBTS: *Clinical Knowledge-Driven Hybrid Transformer With Modality-Correlated Cross-Attention for Brain Tumor Segmentation*. [Preprint DOI](https://doi.org/10.48550/arXiv.2207.07370).
- CMAF-Net: *A Cross-Modal Attention Fusion-Based Deep Neural Network for Incomplete Multi-Modal Brain Tumor Segmentation*. QIMS 14(7), 4579–4604 (2024). [DOI](https://doi.org/10.21037/qims-24-9).
- *Clinical Knowledge-Based Hybrid Swin Transformer for Brain Tumor Segmentation*. CMC 76(3), 3797–3811 (2023). [DOI](https://doi.org/10.32604/cmc.2023.042069).
- MSegNet: *A Multi-View Coupled Cross-Modal Attention Model for Enhanced MRI Brain Tumor Segmentation*. IJ-CIS 18, 63 (2025). [DOI](https://doi.org/10.1007/s44196-025-00787-7).
- CMHF-3DNet: *A Transformer-Based Framework for Improved Brain Tumor Segmentation Across Modalities*. Academic Radiology 33(9), 4042–4059 (2026). [DOI](https://doi.org/10.1016/j.acra.2026.04.045).


## 6. True 2D network and GPU preflight
Four **one-channel** stems encode four MRI slices. A0 takes the masked mean of their
features. A1 adds attention between available modalities at each coarse image location:

\[
Q_m=F_mW_Q,\quad K_n=F_nW_K,\quad V_n=F_nW_V,\qquad
\operatorname{Attention}(Q_m,K_{-m},V_{-m})=
\operatorname{softmax}\!\left(Q_mK_{-m}^{T}/\sqrt{d}\right)V_{-m}.
\]

Queries come from one modality; keys and values come from other available modalities.
Missing modalities are excluded. This follows the cross-attention idea in your sketch.
Here attention is local across modalities, while the 2D U-Net handles spatial processing.
It is not a reproduction of a full spatial or 3D transformer.

The resulting feature map enters a width-16 2D U-Net. Both models use the same CE+Dice
segmentation objective on complete and randomly single-missing input views.
No consistency penalty is used in A0/A1. Output: four tissue-class logits per pixel.

This cell runs a real training-size forward/backward step on your assigned GPU.
Preflight weights are discarded. If it fails due to memory, choose a larger GPU or
reduce batch size for both models in a new study folder.


In [ ]:
workflow.run([sys.executable, "-u", CODE_DIR / "design_network.py", "--self-test"])
PREFLIGHT = workflow.preflight(EDA_DIR, SETTINGS)
workflow.save_json(STUDY_DIR / "gpu_preflight.json", {"environment": ENVIRONMENT, "checks": PREFLIGHT})
print("Preflight passed. This establishes execution, not segmentation accuracy.")


## 7. Train or resume the two-model comparison
STUDY_100_2D runs A0 and A1 for up to 10 epochs, one seed, and validates every two
epochs. Each epoch samples 16 slices per training patient (1,120 slices for 70 patients),
approximately balancing tumor-containing and non-tumor brain slices when available.
Validation and test predictions cover every axial slice, not just tumor slices.

Checkpoints go directly to Drive. Rerunning this cell resumes at the last completed
epoch and skips completed runs. Keep settings, source, data and software versions
fixed. Do not run two sessions against the same folder. After a runtime reset rerun
cells 1–7. No held-out test data are used by training or checkpoint selection.


In [ ]:
TRAIN_COMMAND = workflow.train_command(CODE_DIR, EDA_DIR, STUDY_DIR, SETTINGS)
workflow.run(TRAIN_COMMAND)
EXPERIMENT = json.loads((STUDY_DIR / "models/experiment.json").read_text())
print("Training status:", EXPERIMENT["status"])
print("Selected consistency weights:", EXPERIMENT.get("selected_consistency_weights"))


## 8. Inspect training and validation curves
Use validation to make design choices. Training loss is not held-out segmentation accuracy.


In [ ]:
import matplotlib.pyplot as plt
histories = sorted((STUDY_DIR / "models/final_runs").glob("*/history.json"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for path in histories:
    history = json.loads(path.read_text())
    axes[0].plot([r["epoch"] for r in history], [r["loss"] for r in history], marker=".", label=path.parent.name)
    valid = [r for r in history if "validation_mean_dice" in r]
    axes[1].plot([r["epoch"] for r in valid], [r["validation_mean_dice"] for r in valid], marker=".", label=path.parent.name)
for ax, title in zip(axes, ("Training loss", "Validation mean Dice")):
    ax.set(xlabel="Epoch", title=title)
    ax.grid(alpha=.2)
    if histories:
        ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(STUDY_DIR / "training_curves.png", dpi=160)
plt.show()


## 9. Evaluate after freezing model choices
QUICK_CHECK evaluates its small sample automatically to test the pipeline. For the
100-group study, set RUN_HELD_OUT_TEST=True only after finalizing training choices.

Compare A1−A0 on the same 15 held-out groups across WT/TC/ET and four single-missing
conditions; also report complete-input scores. Patient bootstrap intervals describe
uncertainty for this split and seed; they do not measure variation across training seeds.
Treat this as an exploratory study. Naturally missing MRI files and artificially withheld
modalities are different counts. Both models see the same simulated conditions.

Predicted 2D slices are stacked to evaluate patient-volume Dice and HD95. The network
still has no neighboring-slice input. Set SAVE_ALL_PREDICTIONS=True before evaluation
to retain all hard masks and the default qualitative examples. Interrupted evaluation
restarts in a new directory; completed evaluation is reused.


In [ ]:
RUN_HELD_OUT_TEST = False # @param {type:"boolean"}
SAVE_ALL_PREDICTIONS = False # @param {type:"boolean"}
EVALUATION_DIR = None
if PROFILE == "QUICK_CHECK" or RUN_HELD_OUT_TEST:
    EVALUATION_DIR = workflow.evaluate(CODE_DIR, EDA_DIR, STUDY_DIR, SETTINGS,
                                      save_predictions=(PROFILE == "QUICK_CHECK" or SAVE_ALL_PREDICTIONS))
    print((EVALUATION_DIR / "evaluation_summary.json").read_text())
    import csv
    with (EVALUATION_DIR / "metric_summary.csv").open() as handle:
        for row in csv.DictReader(handle):
            print(row)
    examples = sorted(EVALUATION_DIR.glob("predictions/**/qualitative_comparison.png"))
    for path in examples[:3]:
        display(Image(filename=str(path), width=1000))
else:
    print("Training complete. Held-out test remains unused. Review validation and freeze choices first.")


## 10. Results and next step
Results are saved under STUDY_BASE, separated by profile. Keep `protocol.json`, the
source snapshot, selection manifest, EDA split, checkpoint histories and final metrics.
After QUICK_CHECK succeeds, switch PROFILE to STUDY_100_2D in cell 2 and run cells 2–8.
Then freeze choices and enable held-out evaluation in cell 9.

This is a simpler replacement study, not an identical replication of the original
full-cohort proposal. Report the 100-group selection, 2D inputs, smaller width,
10-epoch budget and single seed in your presentation. More seeds can be added later
under a new frozen protocol. An improvement from cross-attention remains a hypothesis.

Sources: [CMAF-Net](https://qims.amegroups.org/article/view/125898/html),
[PyTorch attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html),
[Colab FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
print("Persistent results:", STUDY_DIR)
print("Profile:", PROFILE)
print("QUICK_CHECK results are pipeline checks only." if PROFILE == "QUICK_CHECK" else "Report full study metrics only after locked evaluation.")



## 11. Optional: test consistency regularization within attention

This is separate from the default two-model study. It reruns A0/A1/A2 in a fresh
`regularization_ablation_lambda0p1` folder with the same full 100-group audit and settings.
A1−A0 tests attention; A2−A1 tests consistency. Lambda=0.1 is a preset chosen before
test evaluation, not a tuned optimum. This requires three training runs.

Select STUDY_100_2D and complete setup/EDA first. Keep both switches false unless
you want to run this optional experiment. Fix this comparison before opening held-out
scores; adding it after viewing test results is exploratory, not an independent confirmation.
The notebook does not change the source code or default study settings to enable this option.


In [ ]:

RUN_REGULARIZATION = False # @param {type:"boolean"}
RUN_REGULARIZATION_TEST = False # @param {type:"boolean"}
REG_SETTINGS = copy.deepcopy(workflow.PROFILES["STUDY_100_2D"])
REG_SETTINGS.update(models=["A0", "A1", "A2"], consistency_weight=0.1,
                    data_source=DATA_SOURCE, local_data_root=LOCAL_DATA_ROOT,
                    expected_groups=100, patient_map=PATIENT_MAP,
                    patient_map_sha256=workflow.digest(PATIENT_MAP) if PATIENT_MAP else None)
REG_DIR = Path(STUDY_BASE) / "regularization_ablation_lambda0p1"
if RUN_REGULARIZATION or RUN_REGULARIZATION_TEST:
    if PROFILE != "STUDY_100_2D" or not EDA_SUMMARY["full_audit"] or EDA_SUMMARY["retained_patient_groups"] != 100:
        raise ValueError("Run the full STUDY_100_2D setup/EDA before this optional comparison.")
    workflow.lock_study(REG_DIR, CODE_DIR, "REGULARIZATION_ABLATION", REG_SETTINGS, ENVIRONMENT)
if RUN_REGULARIZATION:
    checks = workflow.preflight(EDA_DIR, REG_SETTINGS)
    workflow.save_json(REG_DIR / "gpu_preflight.json", {"environment": ENVIRONMENT, "checks": checks})
    workflow.run(workflow.train_command(CODE_DIR, EDA_DIR, REG_DIR, REG_SETTINGS))
    print("Optional training complete. Test evaluation remains a separate choice.")
if RUN_REGULARIZATION_TEST:
    reg_eval = workflow.evaluate(CODE_DIR, EDA_DIR, REG_DIR, REG_SETTINGS,
                                 save_predictions=SAVE_ALL_PREDICTIONS)
    print((reg_eval / "evaluation_summary.json").read_text())
if not RUN_REGULARIZATION and not RUN_REGULARIZATION_TEST:
    print("Optional regularization experiment skipped. Main A0/A1 study remains unchanged.")
